In [ ]:
# Import libraries. You may or may not use all of these.
!pip install -q git+https://github.com/tensorflow/docs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
  # %tensorflow_version only exists in Colab.
  %tensorflow_version 2.x
except Exception:
  pass
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

import tensorflow_docs as tfdocs
import tensorflow_docs.plots
import tensorflow_docs.modeling

In [ ]:
# Import data
!wget https://cdn.freecodecamp.org/project-data/health-costs/insurance.csv
dataset = pd.read_csv('insurance.csv')
dataset.tail()

In [ ]:
# --- 1. Data Cleaning and Conversion ---
# Convert categorical columns to numerical
dataset_encoded = pd.get_dummies(dataset, columns=['sex', 'smoker', 'region'])

# --- 2. Separate Labels (Expenses) from Features ---
labels = dataset_encoded['expenses']
features = dataset_encoded.drop('expenses', axis=1)

# --- 3. Data Split (80% Train, 20% Test) ---
X = features.values
y = labels.values
train_size = int(0.8 * len(X))

train_dataset = X[:train_size]
test_dataset = X[train_size:]
train_labels = y[:train_size]
test_labels = y[train_size:]

# --- 4. Feature Normalization ---
# Convert to float to avoid NumPy errors
train_dataset = train_dataset.astype(np.float64)
test_dataset = test_dataset.astype(np.float64)

mean = train_dataset.mean(axis=0)
std = train_dataset.std(axis=0)
epsilon = 1e-7 # Prevent division by zero

train_dataset = (train_dataset - mean) / (std + epsilon)
test_dataset = (test_dataset - mean) / (std + epsilon)

# --- 5. Build the Model ---
model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=[len(train_dataset[0])]),
    layers.Dense(64, activation='relu'),
    layers.Dense(1)
])

optimizer = tf.keras.optimizers.RMSprop(0.001)

model.compile(loss='mae',
              optimizer=optimizer,
              metrics=['mae', 'mse'])

# --- 6. Train the Model ---
# Train for 100 epochs to ensure accuracy
history = model.fit(train_dataset, train_labels, epochs=100, validation_split=0.2, verbose=1)

In [ ]:
# RUN THIS CELL TO TEST YOUR MODEL. DO NOT MODIFY CONTENTS.
# Test model by checking how well the model generalizes using the test set.
loss, mae, mse = model.evaluate(test_dataset, test_labels, verbose=2)

print("Testing set Mean Abs Error: {:5.2f} expenses".format(mae))

if mae < 3500:
  print("You passed the challenge. Great job!")
else:
  print("The Mean Abs Error must be less than 3500. Keep trying.")

# Plot predictions.
test_predictions = model.predict(test_dataset).flatten()

a = plt.axes(aspect='equal')
plt.scatter(test_labels, test_predictions)
plt.xlabel('True values (expenses)')
plt.ylabel('Predictions (expenses)')
lims = [0, 50000]
plt.xlim(lims)
plt.ylim(lims)
_ = plt.plot(lims,lims)
